In [ ]:
import sys
import os
import json
import pickle
import datetime
from pathlib import Path

from CPTRP_instance_parser_v5 import parse_json_instance
from p_off_utils import compute_p_off, save_p_off_json
from visualize_solutions import Bilevel_Solution_Visualization
from visualize_solutions.Bilevel_Solution_Visualization import summarize_experiment_solutions

# Add the folder containing your modules
module_folder = r"C:\Users\mkoseogl\Desktop\Geneva Data\Cleaned_Codes\heuristics_cleaned_large_noreopt"
if module_folder not in sys.path:
    sys.path.insert(0, module_folder)

import forward_walk_pipeline_poff_clean_noreopt as forward_walk_pipeline_poff_clean
from importlib import reload
reload(forward_walk_pipeline_poff_clean)
from forward_walk_pipeline_poff_clean_noreopt import DualAwareForwardSelection

# ---------------------------------------------------------------------
# Instances
# ---------------------------------------------------------------------
folder_paths = [
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_1669_fix_euclid.json", # 321 (gare) instances ↓ 
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_4579_fix_euclid.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_17828_fix_euclid.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_161979_fix.json",
#r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_297498_fix.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_40548_fix_euclid.json",
r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_71122_fix.json", 
]

# simple baseline mapping (fill with your known values)
# keys should match Path(p).name for entries in folder_paths
baseline_metrics = {
    # example entry taken from current run (replace / add other instances as needed)
    "Instance_1669_fix_euclid.json": {"objective": 168355, "time": 29.080},
    "Instance_4579_fix_euclid.json": {"objective": 369644.16, "time": 116.440},
    "Instance_17828_fix_euclid.json": {"objective": 1108146.19, "time": 602.630},
    "Instance_40548_fix_euclid.json": {"objective": 2149929.36, "time": 398.880},
    "Instance_71122_fix.json": {"objective": 3226631.6, "time": 776.7},
    "Instance_161979_fix.json": {"objective": 0, "time": 4000.0},
    "Instance_297498_fix.json": {"objective": 0, "time": 7200.0}
}


CACHE_DIR = Path(
    r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\cache_parsed"
)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_path_for_json(json_path: str) -> Path:
    p = Path(json_path)
    st = p.stat()
    key = f"{p.stem}__{st.st_size}__{int(st.st_mtime)}.pkl"
    return CACHE_DIR / key


def load_or_parse(json_path: str):
    cpath = cache_path_for_json(json_path)
    if cpath.exists():
        with open(cpath, "rb") as f:
            obj = pickle.load(f)
            print(f"Loading cached instance: {json_path}")
            return obj
    parsed = parse_json_instance(json_path)
    with open(cpath, "wb") as f:
        pickle.dump(parsed, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Parsed and cached instance: {json_path}")
    return parsed


# ---------------------------------------------------------------------
# Hyperparameters
# ---------------------------------------------------------------------
batch_sizes = [20, 50, 70, 100, 200, 300, 500, 700, 1000]     # default grid
mult_A1_set = [0.4, 0.5]
L=10
# ---------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------
for path in folder_paths:
    parsed_data = load_or_parse(path)
    fname = os.path.basename(path)

    baseline_info = baseline_metrics.get(fname, {"objective": None, "time": None})
    baseline_time = baseline_info["time"]

    if baseline_time is not None and baseline_time > 0:
        max_wallclock = float(baseline_time*0.8) # set max wallclock to 80% of baseline time
    else:
        max_wallclock = None  # default max wallclock if no baseline
    # Price upper bounds and disabled arcs (once per instance)
    pinfo = compute_p_off(
        parsed_data,
        P_bar=0.6,
        pt_price=2,
        parking_fee=2.8,
        dep_cost=0.0,
        tol=0.2,
    )
    p_off_map   = pinfo["p_off"]
    global_p_off = pinfo["global_p_off"]
    disable_set  = pinfo["disable"]
    
    #base_save_root = (
    #    r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols"
    #    r"\instanto\nowito\sols_heru\greedy_forward_revision_nosurge"
    #)
    #save_p_off_json(pinfo, base_save_root) 
    

    start_utc = datetime.datetime.now(datetime.timezone.utc)
    print(f"\n=== Instance {fname} | max_wallclock={max_wallclock} | Start UTC: {start_utc} ===\n")

    # -------- loop over mult_A1 values --------
    for mult_A1 in mult_A1_set:
                print(f"\n--- Candidate mult_A1 = {mult_A1} ---")

                # 2) Only now pay the price of building the pipeline + initial model
                pipe = DualAwareForwardSelection(
                    parsed_data,
                    L=L,
                    K_PT=1,
                    mult_A1=mult_A1,
                    H_PT=0,
                    H_A1=10,
                    theta=0.5,
                    P_bar=0.6,
                    P_up=4.0,
                    pt_price=2.0,
                    cost_bike=0.6,
                    parking_fee=2.8,
                    dep_cost=0.0,
                    tolerance=1e-3,
                    quiet=True,
                    threads=4,
                    time_limit_eval=None,
                    max_trial_add=3000,
                    patience_add=3000,
                    p_off_map=p_off_map,
                    global_p_off=global_p_off,
                    disable_arcs=disable_set,
                )

                # heavy step – only once per (instance, mult_A1)
                pipe.build_initial_model_structure()

                # 3) Run forward walk for each effective batch size
                for batch_size in batch_sizes:
                    for mu_station in [1e-6, 1e-10, 0.001, 0.01, 0.1, 0.5]:
                        if fname == "Instance_161979_fix.json" and mu_station == 0.4 and batch_size in [200, 300]:
                            # skip already tested combos for quicker testing
                            continue
                        print(
                            f"\n[Running Instance]: {fname} | mult_A1={mult_A1} | mu_station={mu_station} | "
                            f"batch size {batch_size} | max_wallclock={max_wallclock}\n"
                        )

                        best_val, heuristic_time = pipe.new_run(
                            method="D1_NRC",
                            M=batch_size,
                            mu_station=mu_station,
                            time_limit_eval=6000,
                            max_wallclock = max_wallclock 
                        )

                        # Build save path per (L, mult_A1, batch_size)
                        save_path = rf"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\sols_heru\greedy_forward_revision_nosurge_accept_biger_noreopt_s_zlb\greedy_DAFS_L{L}{mult_A1}_batchsize{batch_size}_mustation{mu_station}"

                        os.makedirs(save_path, exist_ok=True)

                        sol_path = pipe.write_json(
                            json_file_path=path,
                            save_folder=save_path,
                            P_bar=0.6,
                            capacity=None,
                            inventory_level=None,
                            q_tol=0.0,
                        )

                        with open(sol_path, "r", encoding="utf-8") as f:
                            obj = json.load(f)
                        obj["Solution Time (s)"] = heuristic_time
                        with open(sol_path, "w", encoding="utf-8") as f:
                            json.dump(obj, f, indent=4, ensure_ascii=False)

                        end_utc = datetime.datetime.now(datetime.timezone.utc)
                        print(
                            f"\nInstance: {fname}\n"
                            f"Matheuristic Objective: {best_val} | Matheuristic Time(s): {heuristic_time} | batch_size={batch_size} | mult_A1={mult_A1} \n "
                            f"Saved: {sol_path}"
                        )

                        fn = Path(path).name
                        bm = baseline_metrics.get(fn, {"objective": None, "time": None})
                        print(f"Baseline for {fn}: obj={bm['objective']}, time={bm['time']}\n")
                        obj_gap = (100 * (bm["objective"] - best_val) / bm["objective"]) if bm["objective"] else None
                        time_ratio = (100 * (bm["time"] - heuristic_time) / bm["time"]) if bm["time"] else None
                        print(f"Obj Gap:{obj_gap if obj_gap is not None else 'N/A'} | "
                            f"Time Ratio:{time_ratio if time_ratio is not None else 'N/A'}\n")

                        # Save summary to JSON file
                        out_dir = Path(
                            r"C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\sols_heru\greedy_forward_revision_nosurge_accept_biger_noreopt_s_zlb"
                        )
                        out_dir.mkdir(parents=True, exist_ok=True)
                        out_file = out_dir / "summary_results_big_noreopt_smaller_zlb.json"

                        record = {
                            "instance_file": fn,
                            "instance_path": str(path),
                            "mult_A1": mult_A1,
                            "batch_size": batch_size,
                            "mu_station": mu_station,
                            "matheuristic_objective": best_val,
                            "matheuristic_time_s": heuristic_time,
                            "solution_json": str(sol_path),
                            "baseline_objective": bm["objective"],
                            "baseline_time_s": bm["time"],
                            "objective_gap_pct": round(obj_gap, 6) if obj_gap is not None else None,
                            "time_ratio_pct": round(time_ratio, 6) if time_ratio is not None else None,
                            "start_utc": start_utc.isoformat(),
                            "end_utc": end_utc.isoformat(),
                        }

                        try:
                            if out_file.exists():
                                with open(out_file, "r", encoding="utf-8") as f:
                                    try:
                                        data = json.load(f)
                                        if not isinstance(data, list):
                                            data = []
                                    except Exception:
                                        data = []
                            else:
                                data = []

                            data.append(record)
                            with open(out_file, "w", encoding="utf-8") as f:
                                json.dump(data, f, indent=4, ensure_ascii=False)
                            print(f"Saved run summary to {out_file}")
                        except Exception as e:
                            print(f"Failed to save summary JSON: {e}")


                        print(f"End UTC: {end_utc}\n")

# I removed has bike flow condition from the acceptance criteria for big instances, I readded epsilon to trials, lookahead 100, safety=1.05 for large instances
# I removed has bike flow condition from the acceptance criteria for big instances, I removed epsilon to trials, we dont reopt now, lookahead =batchsize, safety=1.05 for large instances



Loading cached instance: C:\Users\mkoseogl\Desktop\Geneva Data\relatota\instos_and_sols\instanto\nowito\Instance_297498_fix.json

=== Instance Instance_297498_fix.json | max_wallclock=5760.0 | Start UTC: 2025-12-30 21:58:44.214013+00:00 ===


--- Candidate mult_A1 = 0.4 ---
Set parameter Username
Academic license - for non-commercial use only - expires 2026-05-21
Set parameter Method to value 1
Set parameter Threads to value 4
<class 'gurobipy.Constr'>
<class 'gurobipy.Constr'>
NumConstrs: 541859

Initial selection stats:
  Total ODs: 270394
  ODs with PT+bike selected: 229077
[init] selection = 16.013s 
[initial-model] build time = 61743.006s

[Running Instance]: Instance_297498_fix.json | mult_A1=0.4 | mu_station=1e-06 | batch size 20 | max_wallclock=5760.0

[batch-forward] using fresh copy of base restricted model
Model Build Time = 3.0011869966983795e-07
Set parameter Method to value -1
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: 1